# 🏥 Tech Challenge — FIAP Pós 9IADT 2026 — Fase 1
## Sistema inteligente de apoio à triagem de mulheres vítimas de violência
---
> **Dados de treino**: Notificações históricas SINAN - 2010 à 2025 (`violencia_BR_25.csv`)  
> **Saída**: Probabilidade de tipo de violência e encaminhamento/exame ser necessário  
---

### 🎯 Modelos comparados neste notebook

| # | Modelo | Tipo | Justificativa |
|---|--------|------|---------------|
| 1 | **Random Forest** | Ensemble (Bagging) | robusto e estável |
| 2 | **XGBoost** | Ensemble (Boosting) | Alta performance em dados tabulares desbalanceados |
| 3 | **Logistic Regression** | Linear | Interpretável, rápido, boa base de comparação |

---

### ⚠️ Aviso Ético
> O sistema é um **apoio à decisão clínica**.
> A avaliação final é sempre responsabilidade do profissional de saúde.  
> Todos os dados tratados conforme **LGPD (Lei nº 13.709/2018)**.

---
## 1️⃣ Importar o dataset
---

In [1]:
import numpy as np
import os
import zipfile
import glob
import pandas as pd
import jinja2

def extract_all_zips(data_dir='data'):
    """Extrai todos os arquivos .zip encontrados em data_dir para a mesma pasta."""
    os.makedirs(data_dir, exist_ok=True)
    zip_paths = glob.glob(os.path.join(data_dir, '*.zip'))
    if not zip_paths:
        print("🔎 Nenhum arquivo .zip encontrado em", data_dir)
        return []
    extracted_files = []
    for zp in zip_paths:
        try:
            with zipfile.ZipFile(zp, 'r') as z:
                z.extractall(data_dir)
                extracted_files.extend(z.namelist())
            print(f"✅ Extraído: {os.path.basename(zp)}")
        except zipfile.BadZipFile:
            print(f"⚠️ Arquivo zip corrompido ou inválido: {zp}")
    return zip_paths


def _read_csv_try(path, max_cols_warn=1000):
    """Tenta ler CSV com combinações comuns de encoding e separador.
       Se o número de colunas for suspeito, emite aviso.
    """
    encodings = ['latin-1', 'utf-8', 'cp1252']
    seps = [',']
    for enc in encodings:
        for sep in seps:
            try:
                df = pd.read_csv(path, sep=sep, encoding=enc, low_memory=False)
                # Checagem de sanidade
                if df.shape[1] > max_cols_warn:
                    print(f"⚠️ {path} lido com {df.shape[1]} colunas (possível separador errado: '{sep}', encoding '{enc}')")
                return df
            except Exception:
                continue
    # última tentativa: deixar o pandas inferir (pode falhar)
    df = pd.read_csv(path, low_memory=False)
    if df.shape[1] > max_cols_warn:
        print(f"⚠️ {path} lido com {df.shape[1]} colunas (pandas inferiu separador/encoding)")
    return df


def load_all_csvs(data_dir='data', concat=False,delete_after_load=False):
    """
    Extrai zips em data_dir, carrega todos os CSVs e retorna:
      - dict_of_dfs: dict {filename: DataFrame}
      - concatenated_df (ou None): DataFrame concatenado se concat=True
    """
    # 1) extrai todos os zips encontrados
    extract_all_zips(data_dir)

    # 2) encontra todos os CSVs (após extração)
    csv_paths = glob.glob(os.path.join(data_dir, '**', '*.csv'), recursive=True)
    if not csv_paths:
        raise FileNotFoundError(f"Nenhum arquivo .csv encontrado em '{data_dir}'.")

    dict_of_dfs = {}
    for p in csv_paths:
        fname = os.path.relpath(p, data_dir)
        try:
            df = _read_csv_try(p)
            dict_of_dfs[fname] = df
            print(f"📥 Carregado: {fname} (linhas: {len(df)}, colunas: {len(df.columns)})")
        except Exception as e:
            print(f"❌ Falha ao carregar {fname}: {e}")

    concatenated_df = None
    if concat and dict_of_dfs:
        try:
            concatenated_df = pd.concat(dict_of_dfs.values(), ignore_index=True, sort=False)
            print(f"🔗 DataFrames concatenados: {len(dict_of_dfs)} arquivos -> {len(concatenated_df)} linhas")
        except Exception as e:
            print(f"⚠️ Não foi possível concatenar os DataFrames automaticamente: {e}")

     # 3) excluir CSVs se solicitado
    deleted_count = 0
    if delete_after_load:
        # Re-obter lista para garantir que pegamos todos os CSVs atuais
        csv_paths_to_delete = glob.glob(os.path.join(data_dir, '**', '*.csv'), recursive=True)
        for p in csv_paths_to_delete:
            try:
                os.remove(p)
                deleted_count += 1
            except Exception as e:
                print(f"⚠️ Não foi possível excluir {p}: {e}")
        print(f"🗑️  Arquivos .csv excluídos: {deleted_count}")            

    return dict_of_dfs, concatenated_df

ModuleNotFoundError: No module named 'jinja2'